# Building a Chatbot with Memory 💬

Here's the single most important thing to understand before building *any* AI chat app:

> **An LLM has no memory.** Every time you call it, it's a completely blank slate. It does *not* remember your previous message.

Sounds broken, right? How does ChatGPT remember your conversation then? The trick is beautifully simple — and it's just a Python **list**. By the end of this notebook you'll know exactly how every chatbot on earth "remembers" things.

This lesson leans entirely on stuff you already know: **lists, dicts, functions** (lesson 1 & 2), the **LLM call** (lesson 4), and **try/except** (lesson 5). Almost nothing new — just a clever idea.

## 1. Prove it: the LLM forgets instantly

Let's catch the model being forgetful. First the usual setup:

In [29]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

Tell it our name in one call...

In [30]:
response = model.invoke("Hi! My name is Dhana. Please remember it.")

print(response.content)

Hi Dhana! While I can't remember personal information or conversations after our chat ends, I'm here to help you with anything you need during our conversation. What can I assist you with today?


...now ask it our name in a **separate** call:

In [31]:
response = model.invoke("What is my name?")
print(response.content)

I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. If you'd like to tell me your name, feel free!


😬 Busted. It has no clue. The LLM has the memory of a goldfish 🐠 — actually worse, because a goldfish at least remembers the last 3 seconds.

**Why?** Each `generate_content` call is independent. The model never saw your first message when answering the second. It's like texting someone who deletes the chat after every single text.

## 2. The fix: keep the whole conversation in a list

The secret: we **remember the conversation ourselves** in a list, and re-send the *entire history* on every call. The model can't remember — so we remind it of everything, every time.

With LangChain, each item in the list is a message object: `HumanMessage` for the user and `AIMessage` for the assistant. `ChatOpenAI` uses these objects to send the conversation in the format expected by OpenAI.

In [32]:
from langchain_core.messages import AIMessage, HumanMessage

conversation = []   # this list IS the chatbot's memory

def chat(user_message):
    messages = [*conversation, HumanMessage(content=user_message)]

    try:
        response = model.invoke(messages)
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

    conversation.extend([
        HumanMessage(content=user_message),
        AIMessage(content=response.content),
    ])
    return response.content

Now let's run the exact same test — name first, then ask:

In [33]:
print(chat("Hi! My name is Dhans. Please remember it."))
print("---")
print(chat("What is my name?"))

Hi Dhans! While I can't remember information between our conversations, I'm here to help you with anything you need right now. What can I assist you with today?
---
Your name is Dhans! How can I assist you today?


🎉 It remembers! Nothing magical happened — we just handed the model the back-story each time.

Let's peek inside its "brain" (our list) to see what's actually stored:

In [34]:
for message in conversation:
    print(message.type, "->", message.content)
    print()

human -> Hi! My name is Dhans. Please remember it.

ai -> Hi Dhans! While I can't remember information between our conversations, I'm here to help you with anything you need right now. What can I assist you with today?

human -> What is my name?

ai -> Your name is Dhans! How can I assist you today?



That growing list is *the entire concept of chatbot memory*. Every chat product you've used is doing this under the hood.

> 💡 **Good to know:** the `google-genai` library also has a shortcut — `client.chats.create(...)` — that manages this list *for* you. We did it by hand on purpose, so you actually understand what that shortcut is hiding. (Remember the classes lesson? Same idea — now you know what's under the hood.)

## 3. Make it talk back: a real chat app

A real chatbot needs to keep taking input in a loop. A `while True` + `input()` loop is clunky inside a notebook, so the proper interactive version lives in **`app.py`** (a Streamlit app) right next to this notebook.

Run it from your terminal:

```bash
streamlit run app.py
```

The app uses the **exact same idea** you just learned — a conversation list that grows with every turn. The only new piece is `st.session_state`, which keeps that list alive between button clicks (otherwise Streamlit would wipe it on every interaction).

Open `app.py`, read it, and tweak it. It's barely 25 lines. 🚀

## 🏋️ Exercise: A movie-recommendation buddy that remembers your taste

Build a chatbot that uses memory to give *personalized* answers.

1. Start a fresh `conversation = []` and reuse the `chat()` function from above.
2. Have this conversation, one `chat()` call at a time:
   - `"I love sci-fi and thrillers, but I can't stand horror movies."`
   - `"Suggest me a movie to watch tonight."`
3. Check that its suggestion respects what you said earlier (sci-fi/thriller, **not** horror). That's memory doing its job!
4. Then ask: `"Why did you pick that one?"` — it should explain using your stated taste.

**Bonus 1:** add a `reset()` function that empties the conversation list — a "new chat" button.

**Bonus 2:** give your bot a personality. Pass a `config` to `generate_content`:
```python
from google.genai import types
config = types.GenerateContentConfig(
    system_instruction="You are a witty movie buff who loves making puns."
)
# ...add config=config to the generate_content call
```

Have fun — try to make it forget on purpose by clearing the list, then watch it ask "who are you?" all over again. 😄

In [41]:
from langchain_core.messages import HumanMessage, SystemMessage


conversation = [SystemMessage(role="system", content="You are a witty movie buff who loves making puns.")]   # this list IS the chatbot's memory


def chat(user_message):
    # 1. add what the user said to memory
    message = [*conversation, HumanMessage(role="user", content=user_message)]

    # 2. send the ENTIRE conversation so far
    try:
        print("Sending the following messages to the model:")
        for msg in message:
            print(f"  {msg.role}: {msg.content}")
        response = model.invoke(message)
        reply = response.content
        conversation.extend([
            HumanMessage(role="user", content=user_message),
            AIMessage(role="assistant", content=reply),
        ])
        return reply
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

def reset_chat():
    global conversation
    conversation = []
    return "Chat history cleared."

print(chat("I love sci-fi and thrillers, but I can't stand horror movies."))
print(chat("Suggest me a movie to watch tonight."))
print(chat("Why did you pick that one?"))

reset_chat()

Sending the following messages to the model:
  system: You are a witty movie buff who loves making puns.
  user: I love sci-fi and thrillers, but I can't stand horror movies.
Sounds like you prefer your thrills with a side of space rather than a side of scares! You might say you’re more into “Star Wars” than “Star Woes.” Just remember, in the world of cinema, there’s no need to be “frightened” by the dark side when you can boldly go where no horror has gone before!
Sending the following messages to the model:
  system: You are a witty movie buff who loves making puns.
  user: I love sci-fi and thrillers, but I can't stand horror movies.
  assistant: Sounds like you prefer your thrills with a side of space rather than a side of scares! You might say you’re more into “Star Wars” than “Star Woes.” Just remember, in the world of cinema, there’s no need to be “frightened” by the dark side when you can boldly go where no horror has gone before!
  user: Suggest me a movie to watch tonight.
Ho

'Chat history cleared.'